<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch06_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>⚙️ Chapter 6 — Data Preprocessing and Feature Engineering</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 75 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Handle missing values with different imputation strategies
- Apply feature scaling: Min-Max, Standard, and Robust normalisation
- Encode categorical variables: one-hot, label, and ordinal encoding
- Engineer new features from existing ones (binning, ratios, log transforms)
- Build a complete preprocessing pipeline from scratch
- Detect and handle outliers using IQR and Z-score methods

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
np.random.seed(42)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Messy Indian job applicant dataset ──────────────────────
n = 200
raw = pd.DataFrame({
    'age':         np.random.randint(21, 55, n).astype(float),
    'experience':  np.random.randint(0, 25, n).astype(float),
    'education':   np.random.choice(['B.Tech', 'M.Tech', 'MBA', 'PhD', None], n),
    'city':        np.random.choice(['Mumbai', 'Bangalore', 'Pune', 'Delhi', 'Hyderabad'], n),
    'salary':      np.random.randint(300000, 2000000, n).astype(float),
    'skills_score': np.random.randint(40, 100, n).astype(float),
    'hired':       np.random.choice([0, 1], n),
})
# Inject missing values and outliers
raw.loc[np.random.choice(n, 20, replace=False), 'age']         = np.nan
raw.loc[np.random.choice(n, 15, replace=False), 'experience']  = np.nan
raw.loc[np.random.choice(n, 10, replace=False), 'skills_score'] = np.nan
raw.loc[np.random.choice(n, 5,  replace=False), 'salary']      = 10000000  # outliers

print('Raw dataset shape:', raw.shape)
print('Missing values:\n', raw.isnull().sum())
raw.head()

## 📖 Section A — Missing Value Handling

| Strategy | When to Use | Code |
|----------|------------|------|
| Drop row | > 30% missing in row | `df.dropna(thresh=...)` |
| Mean fill | Numeric, no outliers | `df.fillna(df.mean())` |
| Median fill | Numeric with outliers | `df.fillna(df.median())` |
| Mode fill | Categorical | `df.fillna(df.mode()[0])` |
| Forward fill | Time-ordered data | `df.fillna(method='ffill')` |

> 💡 **Key Rule:** Never fill with mean if the column has outliers — use median. Outliers inflate the mean, introducing bias into every imputed row.

## 📖 Section B — Feature Scaling

$$\text{Min-Max:} \quad x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

$$\text{Standard (Z-score):} \quad x' = \frac{x - \mu}{\sigma}$$

$$\text{Robust:} \quad x' = \frac{x - \text{median}}{\text{IQR}}$$

| Method | Use When | Sensitive to Outliers? |
|--------|----------|------------------------|
| Min-Max | Neural networks, KNN | Yes |
| Standard | Linear regression, SVM | Somewhat |
| Robust | Data with many outliers | No |

> 💡 **Key Rule:** Tree-based models (Random Forest, XGBoost) do NOT need scaling. Distance-based models (KNN, SVM) and gradient descent models (linear regression, neural networks) DO need it.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: All three scaling methods on salary column
# ─────────────────────────────────────────────────────────────
salary_clean = raw['salary'].dropna()

minmax = (salary_clean - salary_clean.min()) / (salary_clean.max() - salary_clean.min())
standard = (salary_clean - salary_clean.mean()) / salary_clean.std()
q1, q3 = salary_clean.quantile(0.25), salary_clean.quantile(0.75)
robust = (salary_clean - salary_clean.median()) / (q3 - q1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, data, title in zip(axes,
    [salary_clean, minmax, standard, robust],
    ['Original', 'Min-Max [0,1]', 'Standard (Z)', 'Robust (IQR)']):
    ax.hist(data, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(title)
plt.suptitle('Effect of Different Scaling Methods on Salary Distribution', y=1.02)
plt.tight_layout()
plt.show()
print('Shape is preserved; only scale changes. Notice the outliers effect on Min-Max.')

## 📖 Section C — Encoding Categorical Variables

```python
# One-Hot Encoding (no order)
pd.get_dummies(df['city'], drop_first=True)

# Label Encoding (ordered or tree models)
from sklearn.preprocessing import LabelEncoder
# Or manually:
edu_map = {'B.Tech': 0, 'M.Tech': 1, 'MBA': 2, 'PhD': 3}
df['education_encoded'] = df['education'].map(edu_map)

# Ordinal Encoding (explicit ordering)
size_order = ['Small', 'Medium', 'Large']
df['size_enc'] = df['size'].map({v: i for i, v in enumerate(size_order)})
```

> 💡 **Key Rule:** One-hot encode when there is NO ordering (city, colour, department). Use ordinal encoding when there IS ordering (education level, t-shirt size, risk rating).

---
## 🟢 Exercise 6.1 — Impute Missing Values *(Level 1 · Est. 10 min)*

> Impute missing values: age and experience → median; skills_score → mean; education → mode. Assert no nulls remain in these columns.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.1: Impute Missing Values
# ─────────────────────────────────────────────────────────────
df = raw.copy()

# a) Fill age and experience with median
# YOUR CODE HERE

# b) Fill skills_score with mean
# YOUR CODE HERE

# c) Fill education with mode
# YOUR CODE HERE

print('Remaining nulls:', df[['age','experience','skills_score','education']].isnull().sum().sum())

assert df['age'].isnull().sum()         == 0
assert df['experience'].isnull().sum()  == 0
assert df['skills_score'].isnull().sum()== 0
assert df['education'].isnull().sum()   == 0
print('✅ Imputation verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.1 (click to expand)
df = raw.copy()

df['age']          = df['age'].fillna(df['age'].median())
df['experience']   = df['experience'].fillna(df['experience'].median())
df['skills_score'] = df['skills_score'].fillna(df['skills_score'].mean())
df['education']    = df['education'].fillna(df['education'].mode()[0])

print('Remaining nulls:', df[['age','experience','skills_score','education']].isnull().sum().sum())
assert df['age'].isnull().sum() == 0
print('✅ Solution verified! Median for numerics with outliers, mode for categoricals.')

---
## 🟢 Exercise 6.2 — Feature Scaling *(Level 1 · Est. 10 min)*

> Apply Min-Max scaling to `age` and Standard (Z-score) scaling to `skills_score`. Add as new columns. Verify ranges.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.2: Feature Scaling
# ─────────────────────────────────────────────────────────────

# a) Min-Max scale 'age' → 'age_scaled'
df['age_scaled'] = # YOUR CODE HERE

# b) Standard scale 'skills_score' → 'skills_scaled'
df['skills_scaled'] = # YOUR CODE HERE

print(f'age_scaled range : {df.age_scaled.min():.3f} – {df.age_scaled.max():.3f}')
print(f'skills_scaled mean: {df.skills_scaled.mean():.3f}  std: {df.skills_scaled.std():.3f}')

assert abs(df['age_scaled'].min())          < 1e-6, 'Min should be 0'
assert abs(df['age_scaled'].max() - 1.0)    < 1e-6, 'Max should be 1'
assert abs(df['skills_scaled'].mean())      < 0.01, 'Mean should be ~0'
assert abs(df['skills_scaled'].std() - 1.0) < 0.01, 'Std should be ~1'
print('✅ Scaling verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.2 (click to expand)
df['age_scaled']    = (df['age'] - df['age'].min()) / (df['age'].max() - df['age'].min())
df['skills_scaled'] = (df['skills_score'] - df['skills_score'].mean()) / df['skills_score'].std()

print(f'age_scaled   : min={df.age_scaled.min():.3f}  max={df.age_scaled.max():.3f}')
print(f'skills_scaled: mean={df.skills_scaled.mean():.3f}  std={df.skills_scaled.std():.3f}')

assert abs(df['age_scaled'].min()) < 1e-6
print('✅ Solution verified!')

---
## 🟡 Exercise 6.3 — Categorical Encoding *(Level 2 · Est. 15 min)*

> a) One-hot encode 'city' (drop_first=True); b) Ordinal encode 'education' (B.Tech=0, M.Tech=1, MBA=2, PhD=3); c) Assert no original categorical columns remain in a fully numeric DataFrame

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.3: Categorical Encoding
# ─────────────────────────────────────────────────────────────
df2 = df.copy()

# a) One-hot encode 'city'
# YOUR CODE HERE

# b) Ordinal encode 'education'
edu_map = {'B.Tech': 0, 'M.Tech': 1, 'MBA': 2, 'PhD': 3}
# YOUR CODE HERE

# c) Drop remaining categorical columns, assert all numeric
# YOUR CODE HERE

print('Remaining dtypes with object type:', (df2.dtypes == 'object').sum())
print('Shape after encoding:', df2.shape)

assert (df2.dtypes == 'object').sum() == 0, 'All columns must be numeric'
print('✅ Encoding verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.3 (click to expand)
df2 = df.copy()

# One-hot encode city — get_dummies handles this cleanly
df2 = pd.get_dummies(df2, columns=['city'], drop_first=True)

# Ordinal encode education using .map()
edu_map = {'B.Tech': 0, 'M.Tech': 1, 'MBA': 2, 'PhD': 3}
df2['education'] = df2['education'].map(edu_map)

# Any remaining object columns would need encoding — in this case education handled above
print('Object columns remaining:', df2.select_dtypes('object').columns.tolist())
print('Final shape:', df2.shape)

assert (df2.dtypes == 'object').sum() == 0
print('✅ Solution verified! Use one-hot for nominal, ordinal map for ordered categories.')

---
## 🟡 Exercise 6.4 — Feature Engineering *(Level 2 · Est. 15 min)*

> Create new features: a) salary_per_year_exp = salary / (experience + 1); b) is_senior = 1 if experience >= 10 else 0; c) log_salary = log1p(salary); d) age_bin using pd.cut()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.4: Feature Engineering
# ─────────────────────────────────────────────────────────────
fe = df.copy()

# a) Salary per year of experience (add 1 to avoid division by zero)
fe['salary_per_exp'] = # YOUR CODE HERE

# b) Binary flag: is_senior (experience >= 10)
fe['is_senior'] = # YOUR CODE HERE

# c) Log-transform salary (handles right skew)
fe['log_salary'] = # YOUR CODE HERE  np.log1p

# d) Age bins: Young (21-30), Mid (31-40), Experienced (41-55)
fe['age_bin'] = pd.cut(fe['age'], bins=[20, 30, 40, 55],
                       labels=['Young', 'Mid', 'Experienced'])

print(fe[['age', 'age_bin', 'experience', 'is_senior', 'salary', 'log_salary', 'salary_per_exp']].head(8))

assert 'salary_per_exp' in fe.columns
assert fe['is_senior'].isin([0, 1]).all()
assert abs(fe['log_salary'] - np.log1p(fe['salary'])).max() < 1e-9
print('\n✅ Feature engineering verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.4 (click to expand)
fe = df.copy()

fe['salary_per_exp'] = fe['salary'] / (fe['experience'] + 1)
fe['is_senior']      = (fe['experience'] >= 10).astype(int)
fe['log_salary']     = np.log1p(fe['salary'])  # log1p = log(1+x), safe for 0 values
fe['age_bin']        = pd.cut(fe['age'], bins=[20, 30, 40, 55],
                               labels=['Young', 'Mid', 'Experienced'])

print(fe[['age', 'age_bin', 'experience', 'is_senior', 'salary', 'log_salary', 'salary_per_exp']].head(8))

assert fe['is_senior'].isin([0, 1]).all()
print('✅ Solution verified! log1p is your best friend for right-skewed salary/price data.')

---
## 🔴 Exercise 6.5 — Full Preprocessing Pipeline *(Level 3 · Est. 25 min)*

> Write `preprocess(df, target_col)` that: 1) imputes missing values, 2) removes outliers (IQR method), 3) one-hot encodes categoricals, 4) standard-scales numerics, 5) returns (X_array, y_array, feature_names)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 6.5: Full Preprocessing Pipeline
# ─────────────────────────────────────────────────────────────

def preprocess(df, target_col):
    """
    Full ML preprocessing pipeline.
    Returns (X, y, feature_names) as NumPy arrays.
    """
    d = df.copy()

    # Step 1: Impute missing values
    # YOUR CODE HERE

    # Step 2: Remove outliers using IQR on numeric columns
    # YOUR CODE HERE

    # Step 3: One-hot encode all object columns
    # YOUR CODE HERE

    # Step 4: Standard-scale all numeric feature columns
    # YOUR CODE HERE

    # Step 5: Separate features and target
    # YOUR CODE HERE

    return X, y, feature_names

X, y, feats = preprocess(raw, 'hired')
print(f'X shape: {X.shape}  y shape: {y.shape}')
print('Features:', feats[:5], '...')

assert X.shape[1] == len(feats), 'Feature count must match feature names list'
assert len(X) == len(y),         'X and y must have same number of rows'
print('\n✅ Preprocessing pipeline verified!')

In [ ]:
# @title ✅ Solution — Exercise 6.5 (click to expand)

def preprocess(df, target_col):
    d = df.copy()

    # Step 1: Impute
    for col in d.select_dtypes(include='number').columns:
        d[col] = d[col].fillna(d[col].median())
    for col in d.select_dtypes(include='object').columns:
        d[col] = d[col].fillna(d[col].mode()[0])

    # Step 2: Remove outliers via IQR on numeric feature columns (not target)
    numeric_features = [c for c in d.select_dtypes(include='number').columns if c != target_col]
    mask = pd.Series([True] * len(d), index=d.index)
    for col in numeric_features:
        q1, q3 = d[col].quantile(0.25), d[col].quantile(0.75)
        iqr    = q3 - q1
        mask   = mask & (d[col] >= q1 - 1.5 * iqr) & (d[col] <= q3 + 1.5 * iqr)
    d = d[mask].reset_index(drop=True)
    print(f'Rows after outlier removal: {len(d)} (removed {len(df) - len(d)})')

    # Step 3: One-hot encode categoricals
    d = pd.get_dummies(d, drop_first=True)

    # Step 4: Standard scale numeric feature columns
    feature_cols = [c for c in d.columns if c != target_col]
    for col in d[feature_cols].select_dtypes(include='number').columns:
        std = d[col].std()
        if std > 0:
            d[col] = (d[col] - d[col].mean()) / std

    # Step 5: Split
    X = d[feature_cols].values.astype(float)
    y = d[target_col].values
    return X, y, feature_cols

X, y, feats = preprocess(raw, 'hired')
print(f'X shape: {X.shape}  y shape: {y.shape}')
print('Feature names:', feats)
assert X.shape[1] == len(feats)
print('\n✅ Solution verified! This is a production-grade preprocessing pipeline.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: Why do we need feature scaling?</strong></summary>

**Answer:** Many ML algorithms compute distances or gradients that are sensitive to feature magnitude. If salary is in millions and age is in tens, salary will dominate distances in KNN and gradients in linear regression — not because it's more important, but because it's numerically larger. Scaling puts all features on the same scale so the algorithm treats them fairly. Tree-based models don't need scaling because they use thresholds, not distances.
</details>

<details>
<summary><strong>Q2: What is the dummy variable trap and how do you avoid it?</strong></summary>

**Answer:** If you one-hot encode a column with N categories, you get N binary columns that always sum to 1. This creates perfect multicollinearity — one column is always predictable from the others — which causes issues in linear models (matrix inversion fails). Solution: use `drop_first=True` to remove one category (the model learns it as the implicit baseline). Tree models are unaffected by multicollinearity.
</details>

<details>
<summary><strong>Q3: What is feature engineering?</strong></summary>

**Answer:** Feature engineering is creating new input features from existing ones to help the model learn better. Examples: log-transforming a skewed salary column, creating a salary-per-year-of-experience ratio, binning age into groups, combining first and last name length into total name length. Feature engineering is often more impactful than choosing a better algorithm — it's the art of encoding domain knowledge into numbers the model can use.
</details>

<details>
<summary><strong>Q4: How do you detect and handle outliers?</strong></summary>

**Answer:** Detection methods: (1) IQR method — flag values outside [Q1 − 1.5×IQR, Q3 + 1.5×IQR]; (2) Z-score — flag values with |z| > 3; (3) Visual — boxplots and scatter plots. Handling strategies: remove the outlier row (if data entry error), cap to fence (Winsorisation), use a robust model (tree, median-based), or add an 'is_outlier' binary feature. Never blindly delete — understand WHY a value is an outlier first.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 6 Complete!</h3>
<p>You now have a complete preprocessing toolkit:</p>
<ul>
<li>Missing value imputation (median, mean, mode, ffill)</li>
<li>Feature scaling (Min-Max, Standard, Robust)</li>
<li>Categorical encoding (one-hot, ordinal, label)</li>
<li>Feature engineering (ratios, bins, log transforms)</li>
<li>Outlier detection and removal via IQR</li>
<li>A full reusable preprocessing pipeline</li>
</ul>
<p><strong>Next:</strong> Chapter 7 — Supervised Learning: Regression</p>
</div>